In [214]:
!sudo /bin/bash -c "(source /venv/bin/activate; pip install --quiet jupyterlab-vim)"
!jupyter labextension enable

# Imports

In [215]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [216]:
# %%
import logging

import pandas as pd

# /venv/lib/python3.12/site-packages/gspread_pandas/spread.py:401: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)` .replace("", np.nan)
pd.set_option("future.no_silent_downcasting", True)

import helpers.hdbg as hdbg
import helpers.henv as henv
import helpers.hprint as hprint

# hcache.get_global_cache_info()
# hcache.clear_global_cache("all")


# %%
hdbg.init_logger(verbosity=logging.INFO)

_LOG = logging.getLogger(__name__)

_LOG.info("%s", henv.get_system_signature()[0])

hprint.config_notebook()

INFO  # Git
  branch_name='CmampTask11020_Compute_yamm_stats'
  hash='bcf169064'
  # Last commits:
    *   bcf169064 GP Saggese Merge                                                             (   2 hours ago) Sat Jan 18 17:55:42 2025  (HEAD -> CmampTask11020_Compute_yamm_stats, origin/CmampTask11020_Compute_yamm_stats)
    |\  
    | * b53f6a61d Krishna P Taduri CmTask11173_improve_last_name_transformations (#11183)            (   2 hours ago) Sat Jan 18 17:54:46 2025  (origin/master, origin/HEAD, master)
    * | 8c6950cf7 GP Saggese Merge branch 'master' into CmampTask11020_Compute_yamm_stats      (   2 hours ago) Sat Jan 18 17:52:20 2025           
    |\| 
# Machine info
  system=Linux
  node name=9b21dea71c71
  release=6.10.14-linuxkit
  version=#1 SMP Fri Nov 29 17:22:03 UTC 2024
  machine=aarch64
  processor=aarch64
  cpu count=8
  cpu freq=None
  memory=svmem(total=8218251264, available=6600822784, percent=19.7, used=1398374400, free=4820709376, active=1643851776, inactive=700

In [217]:
import gspread

print(gspread.__version__)

import gspread_pandas

print(gspread_pandas.__version__)

# gspread_pandas.conf.get_config()
print(gspread_pandas.conf.get_config()["project_id"])

#!sudo /bin/f bash -c "(source /venv/bin/activate; pip install --upgrade google-api-python-client)"

import importlib

import ck_marketing.process_automation.hyamm as cmprauhy

importlib.reload(hyamm)

import ck_marketing.hunterio.hunter_api as cmhuhuap

importlib.reload(cmhuhuap)

import ck_marketing.linkedin.linkedin_utils as cmliliut

# import helpers.hopenai as hopenai

5.12.4
3.3.0
gspread-gp


# Load data

In [232]:
url = "https://docs.google.com/spreadsheets/d/1kxB15NcHcuhEVtD982qnvbx276J7fmDKvJx63rRpD0E/edit?gid=381241246#gid=381241246"
df = cmprauhy.get_cached_sheet_to_df(url, "List of investors")

# Add column.
columns = df.iloc[0, :].tolist()
columns = [v.split("\n")[0] if "\n" in v else v for v in columns]
columns = [v.strip() for v in columns]
df.columns = columns

# Remove the first row.
df = df.iloc[1:, :]

display(df.head(1))

for col_name in df.columns:
    df[col_name] = df[col_name].str.strip()

# Convert to true/false.
for col_name in df.columns:
    # print("'%s': %s" % (col_name, df[col_name].unique()))
    if col_name in (
        "Name",
        "Email",
        "LinkedIn",
        "Check sizes",
        "Other options",
        "Other types",
        "Other regions",
        "Other sectors",
    ):
        continue
    # #col_name = "Pre-seed"
    hdbg.dassert_is_subset(df[col_name].unique(), ("TRUE", "FALSE"))
    df[col_name] = [(v == "TRUE") for v in df[col_name]]

df.index = range(0, len(df))

# # Split names.
# df.insert(1, "first_name", "")
# df.insert(2, "last_name", "")
# for idx, v in enumerate(df["Name"]):
#     data = v.split()
#     if len(data) > 0:
#         df.loc[idx, "first_name"] = data[0]
#     if len(data) > 1:
#         df.loc[idx, "last_name"] = " ".join(data[1:])
df = cmprauhy.split_first_last_name(df, "Name")

df["origin"] = "super_networking"
cols_map = {
    "origin": None,
    "LinkedIn": "linkedin_url",
    "first_name": None,
    "last_name": None,
    "Email": "email",
}
df = cmprauhy._rename_columns_to_contact_schema(df, cols_map)

#
display(df.head(10))

INFO  Loading cached version from memory ...
INFO  Loading cached version from memory done (0.009 s)


,Name,Email,LinkedIn,Check sizes,Open for more deals from other investors,Ready to share my deal flow with others,Can advise startups,Pre-seed,Seed,Series A,Series B,Series C+,Other options,Angel,Angel Syndicate Lead,VC Fund,Accelerator,Family Office,Private Equity Fund,Venture Studio,Fund of Funds,CVC,Limited Partner,Other types,Globally – everywhere,US,Canada,UK,Europe,Israel,Latin America,Middle East,Africa,Asia Pacific,Other regions,Agnostic – all industries,AI,B2B,B2C,SaaS,Fintech,Healthcare,Biotech,Energy,ClimateTech,E-com & Retail,Future of Work / HRtech,Mobility & Transportation,Marketing / Adtech,PropTech,AgriTech,SpaceTech,Cybersecurity,Blockchain / Crypto,Education,Other sectors
1,Pakpoom Tanthaprabha (Poom),poom@theatlascapital.com,https://www.linkedin.com/in/pakpoom-t-934796110/,100K and above,TRUE,TRUE,TRUE,FALSE,TRUE,TRUE,FALSE,FALSE,,FALSE,FALSE,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,,TRUE,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,TRUE,,FALSE,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,TRUE,TRUE,FALSE,FALSE,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,


,Name,first_name,last_name,email,linkedin_url,Check sizes,Open for more deals from other investors,Ready to share my deal flow with others,Can advise startups,Pre-seed,Seed,Series A,Series B,Series C+,Other options,Angel,Angel Syndicate Lead,VC Fund,Accelerator,Family Office,Private Equity Fund,Venture Studio,Fund of Funds,CVC,Limited Partner,Other types,Globally – everywhere,US,Canada,UK,Europe,Israel,Latin America,Middle East,Africa,Asia Pacific,Other regions,Agnostic – all industries,AI,B2B,B2C,SaaS,Fintech,Healthcare,Biotech,Energy,ClimateTech,E-com & Retail,Future of Work / HRtech,Mobility & Transportation,Marketing / Adtech,PropTech,AgriTech,SpaceTech,Cybersecurity,Blockchain / Crypto,Education,Other sectors,origin
0,Pakpoom Tanthaprabha (Poom),Pakpoom,Tanthaprabha (Poom),poom@theatlascapital.com,https://www.linkedin.com/in/pakpoom-t-934796110/,100K and above,True,True,True,False,True,True,False,False,,False,False,True,False,False,False,False,False,False,False,,True,True,False,False,False,False,False,False,False,True,,False,True,False,False,False,False,False,False,True,True,False,False,True,False,False,False,False,False,False,False,,super_networking
1,Priyaluk (Neuy),Priyaluk,(Neuy),priyaluk_wij@tk-partners.net,https://www.linkedin.com/in/priyaluk-wijitpany...,100K USD min/ 2.5M USD max,True,True,True,False,True,True,False,False,,True,False,True,True,True,False,False,False,False,False,,False,True,False,True,False,False,False,False,False,True,,False,False,False,False,False,False,True,False,True,False,False,False,False,False,True,True,False,False,False,False,,super_networking
2,jihane sadiq,jihane,sadiq,jihane.sadiq@dwtc.com,https://www.linkedin.com/in/jihanesadiq/,1,False,False,True,True,True,True,True,True,,True,True,True,True,True,True,True,True,True,True,DWTC,True,True,True,True,True,True,True,True,True,True,,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,,super_networking
3,Rakesh,Rakesh,,rakesh@sidanaventures.com,https://www.linkedin.com/in/rakeshsidana,100000,True,True,True,True,True,True,False,False,,True,True,False,False,True,False,True,False,False,False,,True,False,False,False,False,False,False,False,False,False,,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,,super_networking
4,Venugopal Sathyanarayana,Venugopal,Sathyanarayana,venu@wizenwelt.com,https://www.linkedin.com/in/svenugopal/,10000 - 1500000,True,True,True,True,True,True,False,False,,False,False,True,False,False,False,False,False,False,False,,False,True,False,False,False,False,False,False,False,True,,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,,super_networking
5,Pankaj Kedia,Pankaj,Kedia,pankaj@2468ventures.vc,https://www.linkedin.com/in/PankajKedia/,"$10K, $100K",True,False,True,True,True,True,False,False,,True,False,True,False,True,False,False,True,False,False,,True,True,True,False,False,True,False,False,False,False,,False,True,True,True,True,False,True,False,False,False,False,True,False,False,False,False,False,True,False,True,,super_networking
6,Alex Burciu,Alex,Burciu,alexbu@gmail.com,https://www.linkedin.com/in/alexbu/,100000,True,True,True,True,True,False,False,False,,True,False,True,False,False,False,False,False,False,True,,False,True,False,False,True,False,False,False,False,False,CEE,True,True,True,False,True,True,False,False,False,False,False,False,False,False,False,False,False,True,True,False,,super_networking
7,Frank Gill,Frank,Gill,frank.gill05@gmail.com,https://www.linkedin.com/in/frankgill-iv/,5000 - 50000,False,False,True,True,True,True,False,False,,True,False,False,False,False,True,False,False,False,False,,False,True,False,False,False,False,False,False,False,False,,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,,super_networking
8,Lou Carpino,Lou,Carpino,loucarpino@pomp

In [240]:
normalize = True
df = cmprauhy.get_data_from_super_networking_gsheet(normalize)

INFO  Loading cached version from memory ...
INFO  Loading cached version from memory done (0.009 s)


In [242]:
# df.iloc[1]

## Clean up names

In [243]:
df = cmliliut.clean_and_track_name_changes(df)

debug_df = cmliliut.get_debug_clean_name_df(df)
cmliliut.get_clean_name_stats(df)

,first_name,last_name,cleaned_first_name,first_alias,cleaned_last_name,second_alias,is_modified
0,Pakpoom,Tanthaprabha (Poom),Pakpoom,,Tanthaprabha,Poom,True
1,Priyaluk,(Neuy),Priyaluk,,,Neuy,True
2,jihane,sadiq,Jihane,,Sadiq,,True


,0
is_modified,56 / 524 = 10.69%
empty_first_name,3 / 524 = 0.57%
empty_last_name,41 / 524 = 7.82%
has_alias,5 / 524 = 0.95%


In [244]:
df = cmprauhy.add_hash(df)
df.head(1)

,Name,first_name,last_name,email,linkedin_url,Check sizes,Open for more deals from other investors,Ready to share my deal flow with others,Can advise startups,Pre-seed,Seed,Series A,Series B,Series C+,Other options,Angel,Angel Syndicate Lead,VC Fund,Accelerator,Family Office,Private Equity Fund,Venture Studio,Fund of Funds,CVC,Limited Partner,Other types,Globally – everywhere,US,Canada,UK,Europe,Israel,Latin America,Middle East,Africa,Asia Pacific,Other regions,Agnostic – all industries,AI,B2B,B2C,SaaS,Fintech,Healthcare,Biotech,Energy,ClimateTech,E-com & Retail,Future of Work / HRtech,Mobility & Transportation,Marketing / Adtech,PropTech,AgriTech,SpaceTech,Cybersecurity,Blockchain / Crypto,Education,Other sectors,origin,cleaned_first_name,first_alias,cleaned_last_name,second_alias,is_modified
hash,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
00032ec8a8c2308a72f3de46329656e6,Leesa Soulodre,Leesa,Soulodre,leesa@r3iventures.com,linkedin.com/in/leesasoulodre,100-2m,False,True,False,False,True,True,False,False,,True,True,True,True,False,False,False,False,True,False,,True,True,True,True,True,True,True,True,True,True,,False,True,True,False,False,False,True,False,True,True,False,False,True,False,False,True,True,True,False,False,,super_networking,Leesa,,Soulodre,,False


In [245]:
# hyamm.save_to_gsheet(df)

In [246]:
# Merge.
df = cmliliut.merge_clean_names_df(df)

In [248]:
df2 = cmprauhy.filter_super_networking_gsheet(df)

mask1= 466
mask2= 404
mask3= 486
mask= 343


In [224]:
display(cmprauhy.head(df2, seed=1, num_rows=10))

shape= (343, 23)
columns= ['Name', 'first_name', 'last_name', 'email', 'linkedin_url', 'Check sizes', 'Seed', 'Series A', 'Globally – everywhere', 'US', 'Agnostic – all industries', 'AI', 'B2B', 'SaaS', 'Fintech', 'Energy', 'ClimateTech', 'E-com & Retail', 'Future of Work / HRtech', 'Mobility & Transportation', 'Marketing / Adtech', 'PropTech', 'AgriTech']



,Name,first_name,last_name,email,linkedin_url,Check sizes,Seed,Series A,Globally – everywhere,US,Agnostic – all industries,AI,B2B,SaaS,Fintech,Energy,ClimateTech,E-com & Retail,Future of Work / HRtech,Mobility & Transportation,Marketing / Adtech,PropTech,AgriTech
hash,,,,,,,,,,,,,,,,,,,,,,,
098ce3e9ff856626d1d5404e92482c9b,Cesar Perez Cardona,Cesar,Perez Cardona,capc1@hotmail.com,https://www.linkedin.com/in/cesar-perez-cardona/,2000 to 10000 dollars,True,False,False,True,True,False,True,True,True,False,False,False,False,False,False,False,True
573fb7d5b1199c3ab49fd339e2abe68a,Priyaluk (Neuy),Priyaluk,,priyaluk_wij@tk-partners.net,https://www.linkedin.com/in/priyaluk-wijitpany...,100K USD min/ 2.5M USD max,True,True,False,True,True,False,False,False,False,True,False,False,False,False,False,True,True
699a2433db1563d8b18d97fc04420b49,Anait Agadzhanian,Anait,Agadzhanian,,https://www.linkedin.com/in/anait/,10k to 100k,True,False,True,False,True,False,False,False,False,True,True,False,False,True,False,False,True
776d5aad57a199cbcd9fae917d6bcb6b,Richard Harris,Richard,Harris,richard@bexleyvc.com,https://www.linkedin.com/in/rick-harris-9a002a...,$100k - $1M,True,False,False,True,True,False,False,False,False,False,True,False,False,False,False,False,True
abc5db410ef294e44546f5313a4d7182,Dennis M. Sponer,Dennis,M Sponer,dsponer@srxadvisors.com,linkedin.com/in/dennismsponer,"100,000 - 100,000,000",True,True,True,False,True,True,True,True,True,True,True,True,True,True,True,False,False
cd270dd2f96ef0847c99912350374699,Mehmet Gonullu,Mehmet,Gonullu,,https://www.linkedin.com/in/mgonullu,TBD,True,False,True,False,True,True,True,True,False,False,False,False,False,False,False,False,False
d90dfff871da5e7548c49bf3bd367836,Stephen Torres,Stephen,Torres,,https://www.linkedin.com/in/stephendtorres,"$5,000-$250,000",True,True,False,True,True,False,False,False,False,False,False,False,False,False,False,False,False
da6fee6936643b91e5a8333eee57e073,Omprakash Karamchandani,Omprakash,Karamchandani,om@themetromaxgroup.com,https://www.linkedin.com/in/omprakashkc?,5k-250k,True,False,False,True,True,False,False,False,False,False,False,False,False,False,False,False,False
e765ccc7c619be2338919cb06dd99b82,Christina Vernali,Christina,Vernali,,https://www.linkedin.com/in/christinavernali/,"$50,000-$100,000",True,False,False,True,True,False,False,False,False,False,False,False,False,False,False,False,False


None

In [249]:
mask = [v != "" for v in df2["email"]]
print(hprint.perc(sum(mask), len(mask)))

mask = [v != "" for v in df2["linkedin_url"]]
print(hprint.perc(sum(mask), len(mask)))

214 / 343 = 62.39%
312 / 343 = 90.96%


In [226]:
display(df2.head())

,Name,first_name,last_name,email,linkedin_url,Check sizes,Seed,Series A,Globally – everywhere,US,Agnostic – all industries,AI,B2B,SaaS,Fintech,Energy,ClimateTech,E-com & Retail,Future of Work / HRtech,Mobility & Transportation,Marketing / Adtech,PropTech,AgriTech
hash,,,,,,,,,,,,,,,,,,,,,,,
00032ec8a8c2308a72f3de46329656e6,Leesa Soulodre,Leesa,Soulodre,leesa@r3iventures.com,linkedin.com/in/leesasoulodre,100-2m,True,True,True,True,True,True,True,False,False,True,True,False,False,True,False,False,True
00c92303ca1c697d89018376533c5022,Ben Dryden,Ben,Dryden,,https://www.linkedin.com/in/ben-dryden-0484591...,$100k - $15m,True,True,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False
00d10773bd3922725205fef6627d86c0,Somesh Surapureddi,Somesh,Surapureddi,,https://www.linkedin.com/in/ssomesh/,$5K -$50K,True,False,False,True,True,True,True,True,True,True,True,False,False,False,False,False,True
0148ac3f5274f68aef7d6a945c9cb806,Jonah Zahnd (Harvard/YFund),Jonah,Zahnd,jzahnd@hbs.edu,https://www.linkedin.com/in/jonah-zahnd-882868...,$100k-$500k,True,False,False,True,True,True,False,True,False,True,True,False,False,True,False,False,False
0407d0ca8c0baadd992e8ac83e42be29,Alina Argasova,Alina,Argasova,,https://www.linkedin.com/in/alinaargasova/,$150K-600K,True,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False


In [227]:
cmprauhy.save_to_gsheet(df2)

https://docs.google.com/spreadsheets/d/1PYp6MqZw2rgM2owLeC9SXbdlyMo4jKZgiXhdYdWKCOo
INFO  Saved to display_tmp
